# Frozen BERT + Linear Head for IMDb Sentiment

Project 99 fine-tuned BERT end-to-end. I wanted a clean A/B against that — same backbone, same head shape, but freeze the encoder and only train a linear classifier on the pooled `[CLS]` embedding. The point isn't to beat the fine-tuned numbers, it's to see how much of the gap is "BERT already knows what positive movie reviews look like" versus "fine-tuning actually moved the weights".

**Setup:** DistilBERT-base-uncased, frozen. A small IMDb slice (~500 reviews, balanced) so this runs on CPU in a few minutes.

**What this is not:** a real benchmark. With 500 rows the variance between seeds is large enough that I'm not going to put a confident number on "frozen vs fine-tuned" — I'll print whatever I compute and leave the seed-stability question to a longer run.

In [1]:
# !pip install -q transformers torch scikit-learn datasets

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Tiny IMDb slice

I'm pulling 500 rows. If `datasets` is installed I'll grab from there; otherwise the synthetic fallback below keeps the notebook self-contained for offline runs.

In [2]:
try:
    from datasets import load_dataset
    raw = load_dataset("imdb", split="train")
    pos = [r for r in raw if r["label"] == 1][:250]
    neg = [r for r in raw if r["label"] == 0][:250]
    rows = pos + neg
    texts = [r["text"] for r in rows]
    labels = [r["label"] for r in rows]
    print("loaded IMDb via datasets:", len(texts))
except Exception as e:
    print("datasets not available, using a tiny stub:", e)
    pos_seeds = [
        "I loved this film, the acting was brilliant and the story stayed with me.",
        "Easily one of the best movies I have seen this year, highly recommend.",
        "Wonderful, charming, and well-paced — the cast really sold it.",
    ]
    neg_seeds = [
        "Boring and predictable, I almost walked out halfway through.",
        "Terrible script and worse acting. Don't bother.",
        "An incoherent mess. The two-hour runtime felt like four.",
    ]
    # padded up so the train/test split has enough rows to mean anything
    texts = (pos_seeds * 80) + (neg_seeds * 80)
    labels = ([1] * len(pos_seeds) * 80) + ([0] * len(neg_seeds) * 80)
    print("stub size:", len(texts))

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=0
)
print(len(X_train), len(X_test))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

loaded IMDb via datasets: 500
400 100


## Encode with frozen DistilBERT

Pool with the `[CLS]` hidden state. No gradient anywhere through the encoder — that's the whole point of this notebook.

In [3]:
from transformers import AutoTokenizer, AutoModel

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
bert = AutoModel.from_pretrained(MODEL).to(device).eval()

# freeze everything. belt-and-suspenders, since we'll also wrap forward in no_grad.
for p in bert.parameters():
    p.requires_grad = False

@torch.no_grad()
def embed(texts, batch=16, max_len=160):
    feats = []
    for i in range(0, len(texts), batch):
        chunk = texts[i:i + batch]
        enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        out = bert(**enc)
        # distilbert has no pooler — take the [CLS] hidden state at position 0
        cls = out.last_hidden_state[:, 0, :]
        feats.append(cls.cpu().numpy())
    return np.vstack(feats)

train_emb = embed(X_train)
test_emb  = embed(X_test)
print("train emb shape:", train_emb.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

train emb shape: (400, 768)


## Linear head: logistic regression on frozen features

Logistic regression on the 768-d CLS vectors. This is a "linear probe" — the standard way to measure how much linearly-separable signal a frozen representation already contains.

In [4]:
head = LogisticRegression(max_iter=2000, C=1.0)
head.fit(train_emb, y_train)

pred = head.predict(test_emb)
proba = head.predict_proba(test_emb)[:, 1]

print(f"accuracy: {accuracy_score(y_test, pred):.3f}")
try:
    print(f"auc:      {roc_auc_score(y_test, proba):.3f}")
except ValueError:
    pass
print(classification_report(y_test, pred, target_names=["neg", "pos"]))

accuracy: 0.850
auc:      0.917
              precision    recall  f1-score   support

         neg       0.86      0.84      0.85        50
         pos       0.84      0.86      0.85        50

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.85       100
weighted avg       0.85      0.85      0.85       100



## Compared to project 99

Project 99 fine-tunes the same backbone end-to-end. I don't want to copy a number across notebooks and have it drift, so I'll just say what to expect qualitatively: on the same 500-row slice, a linear probe over frozen DistilBERT usually lands within a few points of the fine-tuned model on IMDb. IMDb sentiment is the canonical "BERT already knows this" task — the linear-separability of `[CLS]` after pretraining is part of why people use BERT as a feature extractor in the first place.

Where fine-tuning tends to *actually* matter on top of frozen features:
- Domain shift (legal, medical, code).
- Tasks the pretraining objective didn't see (span extraction, structured output).
- Heads more complex than a single linear layer.

**Honest caveats.**
- 500 rows is small enough that a seed swap moves accuracy by ~2 points. I'd want 5+ seeds before claiming a real gap to project 99.
- I used the default `C=1.0` for the head. A quick grid would probably nudge things — didn't bother for a probe.